# 🚣️ NETRA — Pothole Detection Training Pipeline (Kaggle)
## YOLOv8-seg Fine-Tuning on ~10,000 Pothole Images

This notebook trains a YOLOv8 instance-segmentation model on a large pothole dataset using Kaggle's free GPU.

**Steps:**
1. Setup GPU & install dependencies
2. Download & merge pothole datasets (~10K images)
3. Train YOLOv8-seg
4. Evaluate & visualise results
5. Download `best.pt` for NETRA pipeline

---
**⚠️ Before running:**
1. Click **Settings** (right panel) → **Accelerator** → Select **GPU P100** or **GPU T4 x2**
2. Enable **Internet** toggle (right panel → Settings → Internet → On)
3. Click **Run All** or run cells one by one

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install / upgrade dependencies
!pip install -U ultralytics pyyaml --quiet
print("✅ Packages installed")

In [ ]:
import os, shutil, yaml
from pathlib import Path
from ultralytics import YOLO

# Kaggle paths
KAGGLE_INPUT  = Path("/kaggle/input")
KAGGLE_WORK   = Path("/kaggle/working")
DATASET_DIR   = KAGGLE_WORK / "pothole_dataset"
RUNS_DIR      = KAGGLE_WORK / "runs"

print(f"Working dir : {KAGGLE_WORK}")
print(f"Input dir   : {KAGGLE_INPUT}")
print("✅ Imports ready")

## Step 2: Download Pothole Dataset (~10K images)

**Two options** (pick ONE):

### Option A — Attach Kaggle datasets (easiest, no API key needed)
1. Click **+ Add Input** (right panel → Input)
2. Search for **"pothole"** or **"road damage"**
3. Attach these datasets (click "Add"):
   - `chitholian/annotated-potholes-dataset` (~8,500 images)
   - `sovitrath/road-pothole-images-for-pothole-detection` (~1,500 images)
4. Skip the next 2 cells, run the **"Process Kaggle Input datasets"** cell

### Option B — Download from Roboflow (needs free API key)
1. Run the Roboflow cells below

In [ ]:
# ══════════════════════════════════════════════════════════════
#  OPTION A: Attach Kaggle datasets via Input panel (RECOMMENDED)
#  Skip this cell if using Option A — just attach datasets and
#  run the "Process Kaggle Input datasets" cell below.
# ══════════════════════════════════════════════════════════════
#
#  OPTION B: Download from Roboflow
#  Paste your API key below.
# ══════════════════════════════════════════════════════════════
ROBOFLOW_API_KEY = ""  # Only needed for Option B

USE_ROBOFLOW = bool(ROBOFLOW_API_KEY)
print("Mode:", "Roboflow download" if USE_ROBOFLOW else "Kaggle Input datasets")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  Process datasets — works with BOTH Kaggle Input and Roboflow
# ══════════════════════════════════════════════════════════════════════
import cv2

# Prepare merged dataset directory
for split in ("train", "val", "test"):
    (DATASET_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

total_images = 0

# ─── OPTION A: Process Kaggle Input datasets ──────────────────────────
if not USE_ROBOFLOW:
    kaggle_datasets = list(KAGGLE_INPUT.iterdir())
    print(f"Found {len(kaggle_datasets)} attached dataset(s): {[d.name for d in kaggle_datasets]}")

    if len(kaggle_datasets) == 0:
        print("\n❌ No datasets attached!")
        print("   Click '+ Add Input' (right panel) and search for:")
        print("   • 'annotated-potholes-dataset'")
        print("   • 'road-pothole-images-for-pothole-detection'")
        raise RuntimeError("Attach at least one dataset via Input panel")

    img_counter = 0
    for ds_dir in kaggle_datasets:
        print(f"\n📂 Processing: {ds_dir.name}")

        # Find all images recursively
        all_images = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
            all_images.extend(ds_dir.rglob(ext))
            all_images.extend(ds_dir.rglob(ext.upper()))

        # Find all YOLO label files
        all_labels = {f.stem: f for f in ds_dir.rglob("*.txt") if f.name != "classes.txt" and f.name != "README.txt" and f.name != "notes.json"}

        # Check if dataset has YOLO labels or just images
        # Match images to labels
        paired = []
        unpaired_imgs = []
        for img in all_images:
            if img.stem in all_labels:
                paired.append((img, all_labels[img.stem]))
            else:
                unpaired_imgs.append(img)

        print(f"   Images with labels: {len(paired)}")
        print(f"   Images without labels: {len(unpaired_imgs)}")

        # If dataset has YOLO-format subdirs (train/val/test), respect the split
        has_splits = any((ds_dir / s / "images").exists() for s in ("train", "valid", "val", "test"))

        if has_splits:
            print("   Format: YOLO with train/val splits ✅")
            for split_name in ("train", "valid", "val", "test"):
                src_imgs = ds_dir / split_name / "images"
                src_lbls = ds_dir / split_name / "labels"
                if not src_imgs.exists():
                    continue
                dst_split = "val" if split_name == "valid" else split_name
                for img_file in src_imgs.iterdir():
                    if img_file.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"):
                        new_name = f"k{img_counter}_{img_file.name}"
                        shutil.copy2(img_file, DATASET_DIR / dst_split / "images" / new_name)
                        lbl = src_lbls / (img_file.stem + ".txt")
                        if lbl.exists():
                            shutil.copy2(lbl, DATASET_DIR / dst_split / "labels" / f"k{img_counter}_{lbl.name}")
                        img_counter += 1
                        total_images += 1
        elif paired:
            print("   Format: Images + labels (splitting 85/15 train/val)")
            import random
            random.seed(42)
            random.shuffle(paired)
            split_idx = int(len(paired) * 0.85)
            for idx, (img, lbl) in enumerate(paired):
                split = "train" if idx < split_idx else "val"
                new_name = f"k{img_counter}_{img.name}"
                shutil.copy2(img, DATASET_DIR / split / "images" / new_name)
                shutil.copy2(lbl, DATASET_DIR / split / "labels" / f"k{img_counter}_{lbl.name}")
                img_counter += 1
                total_images += 1
        else:
            # Images only — no labels. Skip or warn.
            print("   ⚠️ No YOLO labels found — skipping (need annotated data)")

    print(f"\n✅ Processed {total_images} labelled images from Kaggle datasets")

# ─── OPTION B: Roboflow download ──────────────────────────────────────
else:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)

    # List YOUR workspaces and projects to find the correct slugs
    print("🔍 Listing your Roboflow workspaces...")
    print("   Go to https://universe.roboflow.com/search?q=pothole+detection")
    print("   Find a dataset → click 'Download this Dataset' → YOLOv8 format")
    print("   Copy the workspace/project/version from the download code snippet")
    print()

    # Default: try a well-known public pothole dataset
    # UPDATE these if they don't work — see instructions above
    DATASETS = [
        ("roboflow-100", "pothole-detection-o42ss", 1),
    ]

    for i, (workspace, project_slug, version) in enumerate(DATASETS, 1):
        print(f"[{i}/{len(DATASETS)}] Downloading: {workspace}/{project_slug} v{version}")
        try:
            project = rf.workspace(workspace).project(project_slug)
            ds = project.version(version).download("yolov8", location=str(KAGGLE_WORK / f"temp_ds_{i}"))
            temp_dir = KAGGLE_WORK / f"temp_ds_{i}"

            for split in ("train", "valid", "val", "test"):
                src_imgs = temp_dir / split / "images"
                src_lbls = temp_dir / split / "labels"
                if not src_imgs.exists():
                    continue
                dst_split = "val" if split == "valid" else split
                for img_file in src_imgs.iterdir():
                    if img_file.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
                        new_name = f"rf{i}_{img_file.name}"
                        shutil.copy2(img_file, DATASET_DIR / dst_split / "images" / new_name)
                        lbl = src_lbls / (img_file.stem + ".txt")
                        if lbl.exists():
                            shutil.copy2(lbl, DATASET_DIR / dst_split / "labels" / f"rf{i}_{lbl.name}")
                        total_images += 1
            shutil.rmtree(temp_dir, ignore_errors=True)
            print(f"  ✅ Downloaded {project_slug}")
        except Exception as e:
            print(f"  ❌ Failed: {e}")

if total_images < 100:
    print(f"\n❌ Only {total_images} images. Attach more datasets via '+ Add Input'")
    raise RuntimeError("Insufficient training data")

print(f"\n{'='*50}")
print(f"  Total images: {total_images}")
print(f"{'='*50}")

In [ ]:
# ── Remap all labels to class 0 (pothole) ───────────────────────────
# Different datasets may use different class IDs.
# We remap everything to class 0 = pothole for consistency.
remapped = 0
for split in ("train", "val", "test"):
    lbl_dir = DATASET_DIR / split / "labels"
    if not lbl_dir.exists():
        continue
    for lbl_file in lbl_dir.glob("*.txt"):
        lines = lbl_file.read_text().strip().split("\n")
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                parts[0] = "0"  # remap class to 0
                new_lines.append(" ".join(parts))
        if new_lines:
            lbl_file.write_text("\n".join(new_lines) + "\n")
            remapped += 1

print(f"✅ Remapped {remapped} label files to class 0 (pothole)")

In [ ]:
# ── Create data.yaml for merged dataset ─────────────────────────────
yaml_path = str(DATASET_DIR / "data.yaml")

data_cfg = {
    "path": str(DATASET_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 1,
    "names": ["pothole"],
}

with open(yaml_path, "w") as f:
    yaml.dump(data_cfg, f, sort_keys=False)

# ── Show final stats ──────────────────────────────────────────────
print(f"\n📂 Merged Dataset: {DATASET_DIR}")
print(f"   Classes: pothole (0)")
total = 0
for s in ("train", "val", "test"):
    d = DATASET_DIR / s / "images"
    if d.exists():
        n = len(list(d.iterdir()))
        total += n
        imgs_with_labels = len(list((DATASET_DIR / s / "labels").glob("*.txt")))
        print(f"   {s:6s}: {n:5d} images, {imgs_with_labels:5d} labels")
print(f"   {'Total':6s}: {total:5d} images")
print(f"   data.yaml: {yaml_path}")

assert total > 500, f"❌ Only {total} images found — check dataset downloads"
print(f"\n✅ Dataset ready! ({total} images)")

In [ ]:
# Visualise a few training samples with bounding boxes
import cv2
import matplotlib.pyplot as plt

train_img_dir = DATASET_DIR / "train" / "images"
sample_imgs = sorted(train_img_dir.glob("*"))[:6]

if sample_imgs:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flat, sample_imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        lbl_path = DATASET_DIR / "train" / "labels" / (img_path.stem + ".txt")
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().split("\n"):
                parts = line.strip().split()
                if len(parts) >= 5:
                    _, xc, yc, bw, bh = [float(x) for x in parts[:5]]
                    x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
                    x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    cv2.putText(img, "pothole", (x1, y1 - 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")

    plt.suptitle("Training Samples (merged dataset)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No training images found to visualize")

## Step 3: Train YOLOv8-seg

**Model options** (change `MODEL_SIZE` below):
- `yolov8n-seg.pt` — nano: fastest, lowest accuracy
- `yolov8s-seg.pt` — **small: best speed/accuracy trade-off** ← default
- `yolov8m-seg.pt` — medium: best accuracy, slower

**Kaggle GPU limits:**
- P100: 16 GB VRAM — batch 16 works for all model sizes
- T4: 16 GB VRAM — batch 16 works, use 8 for yolov8m-seg
- Session limit: **30 hours/week** (use wisely!)

In [ ]:
# ════════════════════════════════════════════════
#  TRAINING CONFIG — adjust these as needed
# ════════════════════════════════════════════════
MODEL_SIZE = "yolov8s-seg.pt"   # Best speed/accuracy trade-off
EPOCHS     = 100                 # 100 is good for ~10K images
BATCH_SIZE = 16                  # Reduce to 8 if OOM on T4
IMG_SIZE   = 640
PATIENCE   = 30                  # Early stopping patience
# ════════════════════════════════════════════════

print(f"Model    : {MODEL_SIZE}")
print(f"Epochs   : {EPOCHS}")
print(f"Batch    : {BATCH_SIZE}")
print(f"Img size : {IMG_SIZE}")
print(f"Patience : {PATIENCE}")

In [ ]:
# Load pretrained YOLOv8-seg and start training
model = YOLO(MODEL_SIZE)

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,
    project=str(RUNS_DIR),
    name="netra-pothole",
    exist_ok=True,
    plots=True,

    # Augmentation (tuned for road/dashcam scenes)
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.1,
    erasing=0.4,

    # Segmentation
    overlap_mask=True,

    # LR schedule
    cos_lr=True,
    lr0=0.01,
    lrf=0.005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    weight_decay=0.0005,
    optimizer="SGD",

    # Mosaic warm-down (disable last 15 epochs for cleaner convergence)
    close_mosaic=15,

    # Saving
    save_period=25,
    patience=PATIENCE,
    verbose=True,
)

run_dir = str(RUNS_DIR / "netra-pothole")
print(f"\n✅ Training complete! Results in: {run_dir}")

## Step 4: Evaluate Results

In [ ]:
# Training curves
from IPython.display import Image, display

results_img = f"{run_dir}/results.png"
if os.path.exists(results_img):
    print("📈 Training Results:")
    display(Image(filename=results_img, width=900))
else:
    print("⚠️ results.png not found")

In [ ]:
# Confusion matrix
cm_img = f"{run_dir}/confusion_matrix.png"
if os.path.exists(cm_img):
    print("📊 Confusion Matrix:")
    display(Image(filename=cm_img, width=500))
else:
    print("⚠️ confusion_matrix.png not found")

In [ ]:
# Validation predictions
for name in ("val_batch0_pred.jpg", "val_batch1_pred.jpg", "val_batch2_pred.jpg"):
    path = f"{run_dir}/{name}"
    if os.path.exists(path):
        print(f"\n🔎 {name}:")
        display(Image(filename=path, width=800))

In [ ]:
# Detailed validation metrics
best_pt = f"{run_dir}/weights/best.pt"
best_model = YOLO(best_pt)
metrics = best_model.val(data=yaml_path, imgsz=IMG_SIZE, device=0)

print(f"\n{'='*52}")
print("  NETRA Model Evaluation")
print(f"{'='*52}")
print(f"  Box  mAP@50    : {metrics.box.map50:.4f}")
print(f"  Box  mAP@50-95 : {metrics.box.map:.4f}")
print(f"  Box  Precision : {metrics.box.mp:.4f}")
print(f"  Box  Recall    : {metrics.box.mr:.4f}")
if hasattr(metrics, "seg") and metrics.seg is not None:
    print(f"  ────────────────────────────────────────")
    print(f"  Mask mAP@50    : {metrics.seg.map50:.4f}")
    print(f"  Mask mAP@50-95 : {metrics.seg.map:.4f}")
print(f"{'='*52}")

# Quality check
if metrics.box.map50 > 0.5:
    print("\n✅ Model looks good! mAP@50 > 0.5")
elif metrics.box.map50 > 0.3:
    print("\n⚠️ Decent model. Consider more epochs or larger model size.")
else:
    print("\n❌ Low mAP. Check dataset quality or increase training epochs.")

## Step 5: Download Model

Download `best.pt` and place it in your NETRA project:
```
NETRA/
  weights/
    best.pt   ← put it here
```
Then run: `python demo.py`

In [ ]:
# Copy best.pt to /kaggle/working/ for easy download
best_pt = f"{run_dir}/weights/best.pt"
download_path = str(KAGGLE_WORK / "best.pt")

if os.path.exists(best_pt):
    shutil.copy2(best_pt, download_path)
    size_mb = os.path.getsize(download_path) / 1e6
    print(f"📦 Model size: {size_mb:.1f} MB")
    print(f"📥 Saved to: {download_path}")
    print(f"")
    print(f"To download from Kaggle:")
    print(f"  1. Click 'Save Version' (top-right) -> 'Save & Run All' -> 'Save'")
    print(f"  2. Go to your notebook's Output tab")
    print(f"  3. Download best.pt")
    print(f"")
    print(f"Then copy best.pt -> NETRA/weights/best.pt")
    print(f"And run: python demo.py")
else:
    print("❌ best.pt not found! Training may have failed.")
    print("  Check the training logs above for errors.")

In [ ]:
# Also save last.pt (for resuming training if needed)
last_pt = f"{run_dir}/weights/last.pt"
if os.path.exists(last_pt):
    shutil.copy2(last_pt, str(KAGGLE_WORK / "last.pt"))
    print("✅ last.pt also saved (for resuming training)")

# Save training results image
results_img = f"{run_dir}/results.png"
if os.path.exists(results_img):
    shutil.copy2(results_img, str(KAGGLE_WORK / "training_results.png"))
    print("✅ training_results.png saved")

print(f"\n{'='*50}")
print(f"  DONE! All outputs saved to /kaggle/working/")
print(f"  Click 'Save Version' to access downloads.")
print(f"{'='*50}")